# TP2 — Generation : des passages pertinents à la réponse — **corrigé**

*Version corrigée : identique au TP, avec les exercices résolus (les cellules appellent l'API : non pré-exécutées).*

**Objectif de l'après-midi** : ajouter le **G** de RAG. Les k chunks trouvés au TP1 sont donnés à un LLM,
avec la question, pour qu'il rédige une réponse **appuyée sur les documents et vérifiable**.

```
question ──► search (TP1) ──► k chunks ──┐
                                          ├──► prompt ──► LLM ──► réponse + sources
consignes (system) + question ────────────┘
```

Plan :
1. Appeler un LLM via une API
2. Sans RAG : le LLM seul
3. Avec RAG
4. Expériences
5. Exercices

Prérequis : l'index du TP1 (`index/`) et la clé d'API dans `.env` (voir `README.md`).

In [1]:
import time

from dotenv import load_dotenv
from litellm import completion

import settings
from rag import load_index, search, build_prompt, generate, answer

load_dotenv()  # charge la clé d'API (LLM_API_KEY) depuis .env
chunks, embeddings = load_index()  # l'index construit au TP1
print(len(chunks), "chunks,", embeddings.shape[1], "dimensions")

601 chunks, 384 dimensions


## 1. Appeler un LLM via une API

Un LLM de plusieurs milliards de paramètres tourne sur des GPU distants : on l'appelle via une **API HTTP**.
Tous les fournisseurs (OpenAI, Mistral, Anthropic, Hugging Face, …) ont adopté le même format de conversation :
une liste de **messages**, chacun avec un rôle :

- `system` : les consignes données au modèle (ton, format, limites) ;
- `user` : ce que dit l'utilisateur ;
- `assistant` : ce que répond le modèle.

La librairie **LiteLLM** expose une seule fonction, `completion()`, quel que soit le fournisseur.
Ici, on passe par une **API de Cross Data**, compatible OpenAI : le modèle est choisi dans `settings.GENERATION_MODEL`,
la clé d'API est lue dans `.env`.

In [2]:
messages = [
    {"role": "system", "content": "Tu es un assistant concis. Réponds en deux phrases maximum."},
    {"role": "user", "content": "Qu'est-ce qu'un capteur à ultrasons ?"},
]

reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0)

print(reponse.choices[0].message.content)
print("\nusage :", reponse.usage)

Un capteur à ultrasons mesure la distance en émettant des ondes sonores inaudibles et en calculant le temps de retour de l'écho. Il est souvent utilisé pour la détection d'obstacles ou la mesure de niveaux.

usage : Usage(completion_tokens=52, prompt_tokens=43, total_tokens=95, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), service_tier='standard')


`usage` donne le nombre de tokens **envoyés** (`prompt_tokens`) et **générés** (`completion_tokens`) : c'est ce qui est facturé,
et c'est ce qui doit tenir dans la fenêtre de contexte du modèle.

La **température** règle le hasard dans le choix de chaque token : 0 = le token le plus probable à chaque fois ; 2 = plus de variété.

In [3]:
messages = [{"role": "user", "content": "Propose un nom original pour un robot explorateur. Réponds par le nom seul."}]

for temperature in (0.0, 0.0, 1.5, 1.5):
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=temperature, max_tokens=20)
    print(f"température {temperature} → {reponse.choices[0].message.content.strip()}")

température 0.0 → Nixar
température 0.0 → Nixar
température 1.5 → Nixis
température 1.5 → **Terra-X-17**


## 2. Sans RAG : le LLM seul

Posons une question précise sur nos datasheets, **sans lui donner les documents**.

In [4]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

reponse = completion(model=settings.GENERATION_MODEL, messages=[{"role": "user", "content": question}], temperature=0)
print(reponse.choices[0].message.content)

Le **BME280** (capteur de pression, température et humidité de Bosch) a une consommation de courant très faible en **mode veille (sleep mode)**, ce qui en fait un choix idéal pour les applications alimentées par batterie.

### **Consommation en mode veille (Sleep Mode) :**
- **Typique** : **~0,1 µA** (microampère)
- **Maximale** : **~0,5 µA** (selon les conditions et la tension d'alimentation)

### **Comparaison avec d'autres modes :**
| Mode               | Consommation typique | Temps d'activation |
|--------------------|----------------------|--------------------|
| **Sleep Mode**     | **0,1 µA**           | -                  |
| **Forced Mode**    | **~1,8 µA** (mesure) | ~1 ms              |
| **Normal Mode**    | **~3,6 µA** (continu)| -                  |

### **Remarques importantes :**
1. **Le BME280 ne consomme presque rien en veille**, ce qui permet une longue durée de vie des batteries.
2. **En mode forcé (Forced Mode)**, il s'allume brièvement pour une mesure, puis retou

**À observer**

Le modèle répond avec assurance… mais a-t-il raison ? Il n'a que ce qu'il a mémorisé pendant son entraînement
(peut-être cette datasheet, peut-être pas, peut-être une version différente). Rien ne permet de le vérifier :
c'est le problème des **hallucinations**. Vérifiez dans `data/bme280.pdf` (chapitre "Electrical characteristics").

## 3. Avec RAG

Trois étapes :
1. **chercher** les chunks pertinents (TP1 : `search`) ;
2. **construire le prompt** : consignes + extraits + question (`build_prompt`) ;
3. **générer** la réponse (`generate`).

Regardons d'abord le prompt réellement envoyé au modèle.

In [5]:
extraits = search(question, chunks, embeddings, k=settings.TOP_K)   # 1. retrieval
messages = build_prompt(question, extraits)                         # 2. prompt

print("=== system ===")
print(messages[0]["content"])
print("\n=== user (début) ===")
print(messages[1]["content"][:1500], "\n[...]")

=== system ===
Tu es un assistant technique qui répond à des questions sur des datasheets de composants électroniques.
Réponds UNIQUEMENT à partir des extraits fournis. Si l'information n'y figure pas, dis-le clairement.
Réponds dans la langue de la question.

=== user (début) ===
Extraits de documentation :

[esp32.pdf, page 31]
4 Functional Description ESP32-S0WD (NRND) has a maximum CPU frequency of 160 MHz. • When Wi-Fi is enabled, the chip switches between Active and Modem-sleep modes. Therefore, power consumption changes accordingly. • In Modem-sleep mode, the CPU frequency changes automatically. The frequency depends on the CPU load and the peripherals used. • During Deep-sleep, when the ULP coprocessor is powered on, peripherals such as GPIO and RTC I2C are able to operate. • When the system works in the ULP sensor-monitored pattern, the ULP coprocessor works with the ULP sensor periodically and the ADC works with a duty cycle of 1%, so the power consumption is 100 µA. 4.3.2 Ul

In [6]:
print(generate(messages))                                           # 3. generation

D'après l'extrait fourni :

[bme280.pdf, page 3]
"When the sensor is disabled, current consumption drops to 0.1 µA."

Cela correspond au mode veille (sleep mode) du BME280.

**Réponse :**
La consommation de courant du BME280 en mode veille (sleep mode) est de **0,1 µA**.


**À observer**

- La connaissance n'est plus dans le modèle : elle est **dans le prompt**.
- La réponse cite `[fichier, page]`. On peut donc **vérifier** les réponses. C'est le principal intérêt du RAG en entreprise.
- Le prompt système impose des règles : répondre uniquement à partir des extraits, dire quand l'information n'y est pas, citer.

`answer` dans `rag.py` enchaîne les trois étapes et renvoie la réponse **et** les chunks utilisés.

In [7]:
questions = [
    "Quelle est la plage de tension d'alimentation (VSYS) du Raspberry Pi Pico ?",
    "Quelle est la fréquence maximale du processeur de l'ESP32 ?",
    "Quelle est la précision du LM35 à 25 °C ?",
]

for q in questions:
    reponse, utilises = answer(q, chunks, embeddings)
    print("###", q)
    print(reponse)
    print("chunks utilisés :", ", ".join(f"{c['source']} p.{c['page']}" for c in utilises))
    print()

### Quelle est la plage de tension d'alimentation (VSYS) du Raspberry Pi Pico ?
D'après les extraits fournis, la plage de tension d'alimentation **VSYS** du Raspberry Pi Pico n'est pas explicitement mentionnée.

Cependant, on sait que :
- **VSYS** est la tension principale d'entrée du système (page 21).
- Le RP2040 et ses E/S fonctionnent à **3,3 V** (page 4 et 20).
- Le convertisseur buck-boost **RT6150** génère une sortie fixe de **3,3 V** à partir de **VSYS** (page 21).
- Il est possible d'alimenter le Pico via **VSYS** avec une tension externe comprise entre **~2,3 V et 5,5 V** (page 22).

**Réponse :**
La plage de tension d'alimentation **VSYS** du Raspberry Pi Pico est d'environ **2,3 V à 5,5 V**.
chunks utilisés : pico.pdf p.11, pico.pdf p.7, pico.pdf p.4, pico.pdf p.31, pico.pdf p.20, pico.pdf p.3, pico.pdf p.15, pico.pdf p.25, pico.pdf p.29, pico.pdf p.10, pico.pdf p.6, pico.pdf p.21, pico.pdf p.24, pico.pdf p.16, pico.pdf p.22

### Quelle est la fréquence maximale du processe

Et si la réponse n'est **pas** dans les documents ? Un bon RAG doit le dire plutôt qu'inventer.

In [8]:
reponse, utilises = answer("Quel est le prix du Raspberry Pi Pico ?", chunks, embeddings)
print(reponse)

D'après les extraits fournis, le prix du **Raspberry Pi Pico** est de **US$4.00** (RRP) par unité.

*(Source : Tableau A.2, page 25 du pico.pdf)*


## 4. Expériences

### 4.a Combien de chunks donner au modèle ?

`k` (dans `settings.TOP_K`) est un compromis : plus de chunks = plus de chances que la réponse y soit,
mais plus de tokens (coût, latence), plus de bruit pour le modèle, et une fenêtre de contexte plus vite atteinte.

In [9]:
question = "Quelle est la consommation de courant du BME280 en mode veille (sleep mode) ?"

for k in (1, 5, 15):
    messages = build_prompt(question, search(question, chunks, embeddings, k=k))
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0, max_tokens=200)
    print(f"k={k:<3} tokens envoyés : {reponse.usage.prompt_tokens:<6} → {reponse.choices[0].message.content[:160]!r}")

k=1   tokens envoyés : 342    → "D'après les extraits fournis, la consommation de courant du BME280 en mode veille (sleep mode) n'est pas mentionnée."
k=5   tokens envoyés : 1142   → 'D\'après l\'extrait fourni :\n\n[bme280.pdf, page 3]\n"When the sensor is disabled, current consumption drops to 0.1 µA."\n\nEn mode veille (sleep mode), la consommati'
k=15  tokens envoyés : 3900   → 'D\'après l\'extrait fourni :\n\n[bme280.pdf, page 3]\n"When the sensor is disabled, current consumption drops to 0.1 µA."\n\nCette information correspond au mode veill'


### 4.b Avec ou sans RAG : même modèle, deux réponses

On garde le même modèle (`settings.GENERATION_MODEL`) et on lui pose la même question deux fois : sans les documents,
puis avec les extraits trouvés par la recherche. Comparez la durée, les tokens consommés et surtout le contenu.

In [10]:
essais = [
    ("sans RAG", False),
    ("avec RAG", True),
]

for nom, avec_rag in essais:
    if avec_rag:
        messages = build_prompt(question, search(question, chunks, embeddings))
    else:
        messages = [{"role": "user", "content": question}]
    debut = time.time()
    reponse = completion(model=settings.GENERATION_MODEL, messages=messages, temperature=0, max_tokens=200)
    duree = time.time() - debut
    print(f"### {nom} — {duree:.1f} s, {reponse.usage.total_tokens} tokens")
    print(reponse.choices[0].message.content.strip(), "\n")

### sans RAG — 2.6 s, 236 tokens
Le **BME280** (Bosch Sensortec) est un capteur combiné de pression, température et humidité. En **mode veille (sleep mode)**, sa consommation de courant est extrêmement faible, car la plupart des circuits internes sont désactivés.

### **Consommation en mode veille (sleep mode) :**
- **Typique** : **~0,1 µA** (microampère)
- **Maximale** : **~1 µA** (selon les conditions et la tension d'alimentation)

### **Comparaison avec d'autres modes :**
| Mode               | Consommation typique |
|--------------------|----------------------|
| **Sleep Mode**     | **0,1 µA**           |
| **Forced Mode**    | ~1 mA (mesure unique)|
| **Normal Mode**    | ~3 mA (mesures continues)|

### **Remarques importantes :**
1. **Tension d 

### avec RAG — 1.6 s, 3984 tokens
D'après l'extrait fourni :

[bme280.pdf, page 3]
"When the sensor is disabled, current consumption drops to 0.1 µA."

Cette information correspond au mode veille (sleep mode) du BME280.

**Réponse :**
L

**À observer**

- Avec RAG, la requête coûte plus de tokens (les extraits sont dans le prompt), mais la réponse cite sa source :
  on peut la vérifier. Sans RAG, la réponse est peut-être juste, mais rien ne permet de le savoir.
- Un petit modèle **bien alimenté en contexte** fait souvent aussi bien qu'un gros modèle livré à sa mémoire,
  pour une fraction du coût, de la latence et de l'énergie. C'est l'argument de **frugalité** du RAG :
  la connaissance est dans les documents (gratuits à lire), l'intelligence du modèle ne sert qu'à la restituer.

**Et avec un gros modèle ?** Vous pouvez faire l'expérience à la main avec un
assistant grand public (ChatGPT, Mistral, Claude…) :

1. posez-lui la même question, sans document ;
2. importez la datasheet complète (`data/bme280.pdf`) dans la conversation, puis reposez la question.

Ces modèles sont bien plus gros que le nôtre, et leur fenêtre de contexte peut contenir une datasheet entière.
Comparez avec notre RAG : la réponse est-elle meilleure ? 

*Nos datasheets sont publiques. En entreprise, vérifiez avant d'importer un document dans un outil grand public :
ce que vous y déposez quitte votre système d'information.*

## 5. Exercices

Tout se fait dans les cellules ci-dessous ; `rag.py` reste inchangé.

### Exercice 1 — le prompt système

La cellule est complète et s'exécute telle quelle : l'exercice consiste à **modifier le texte de `SYSTEM_PROMPT`**
(format de réponse — liste, tableau —, longueur, unités, comportement quand la réponse est absente, position des
citations…) et à observer l'effet sur les réponses.

In [11]:
SYSTEM_PROMPT = """Tu réponds à des questions techniques à partir d'extraits de datasheets.
Règles :
- Donne d'abord la valeur demandée avec son unité, puis une phrase d'explication au maximum.
- Utilise UNIQUEMENT les extraits fournis. Si l'information n'y est pas, réponds exactement : "Information absente des extraits fournis."
- Termine par la source : [fichier, page N].
- Réponds dans la langue de la question."""


def answer_avec_consignes(question, chunks, embeddings, system_prompt, k=settings.TOP_K):
    # Comme rag.answer, mais le prompt système est passé en paramètre
    extraits = search(question, chunks, embeddings, k)
    messages = build_prompt(question, extraits)
    messages[0] = {"role": "system", "content": system_prompt}
    return generate(messages), extraits


for q in ["Quelle est la précision du LM35 à 25 °C ?", "Quel est le prix du Raspberry Pi Pico ?"]:
    reponse, _ = answer_avec_consignes(q, chunks, embeddings, SYSTEM_PROMPT)
    print("###", q)
    print(reponse)
    print()

### Quelle est la précision du LM35 à 25 °C ?
±0.2 °C à 25 °C pour le LM35A, ±0.4 °C pour le LM35.

[lm35.pdf, page 5]

### Quel est le prix du Raspberry Pi Pico ?
4.00 US$

Prix de vente au détail (RRP) du Raspberry Pi Pico standard.

[pico.pdf, page 25]



**Correction, exercice 1** — pas de solution unique : celle-ci impose la valeur + unité en tête, une explication
au maximum, et surtout une **phrase de refus standardisée** — dont l'intérêt ressort au bonus 8 du TP3 (questions pièges) :
un refus toujours identique devient trivial à détecter automatiquement.

### Exercice 2 — la recherche hybride dans le RAG

La cellule importe `search_hybrid` depuis le **corrigé du TP1** (`solutions/tp1.py`, fourni en fin de TP1) :
pas besoin d'avoir terminé le TP1 pour faire cet exercice. Vous préférez vos propres fonctions ? Collez-les sous
l'import : elles remplacent celle du corrigé.

Exécutez la cellule : les questions sur des identifiants exacts (`ctrl_meas`…) s'améliorent-elles ?

In [12]:
import re

import numpy as np
from rank_bm25 import BM25Okapi

try:
    from solutions.tp1 import search_hybrid  # corrigé du TP1
except ModuleNotFoundError:
    print("Corrigé du TP1 absent (solutions/tp1.py) : collez vos fonctions ci-dessous.")

# Collez ici vos fonctions du TP1 si vous préférez les vôtres : MOTS_VIDES, tokenize_sans_mots_vides,
# search_bm25_sans_mots_vides et search_hybrid.


def answer_hybride(question, chunks, embeddings, k=settings.TOP_K):
    # Comme rag.answer, mais le retrieval est hybride au lieu de purement sémantique
    extraits = search_hybrid(question, chunks, embeddings, k)
    return generate(build_prompt(question, extraits)), extraits


for q in ["À quelle adresse se trouve le registre ctrl_meas du BME280 ?",
          "Quelle est la tension d'alimentation du HC-SR04 ?"]:
    reponse, utilises = answer_hybride(q, chunks, embeddings)
    print("###", q)
    print(reponse)
    print("chunks utilisés :", ", ".join(f"{c['source']} p.{c['page']}" for c in utilises))
    print()

### À quelle adresse se trouve le registre ctrl_meas du BME280 ?
D'après les extraits fournis, le registre **ctrl_meas** du BME280 se trouve à l'adresse **0xF4**.

*(Source : [bme280.pdf, page 27, Table 18])*.
chunks utilisés : bme280.pdf p.27, bme280.pdf p.27, bme280.pdf p.3, bme280.pdf p.1, bme280.pdf p.32, bme280.pdf p.34, bme280.pdf p.29, bme280.pdf p.33, esp32.pdf p.27, bme280.pdf p.28, bme280.pdf p.25, bme280.pdf p.5, lm35.pdf p.1, bme280.pdf p.55, bme280.pdf p.35

### Quelle est la tension d'alimentation du HC-SR04 ?
D'après l'extrait fourni pour le **HC-SR04** (hcsr04.pdf, page 1), la tension d'alimentation est :

**DC 5 V**
chunks utilisés : esp32.pdf p.53, hcsr04.pdf p.1, bme280.pdf p.11, esp32.pdf p.10, bme280.pdf p.11, esp32.pdf p.9, bme280.pdf p.31, esp32.pdf p.9, bme280.pdf p.17, esp32.pdf p.8, esp32.pdf p.69, esp32.pdf p.7, lm35.pdf p.15, esp32.pdf p.7, lm35.pdf p.14



**Correction, exercice 2** — résultat attendu : ces deux questions, ratées par la recherche purement sémantique,
obtiennent maintenant les bons chunks (`bme280.pdf` p.27-29, `hcsr04.pdf` p.1) — et donc une réponse correcte et sourcée.

### Exercice 3 — réécrire la question avant la recherche

Les datasheets sont en anglais, les questions en français. Complétez `rewrite_query` : demander au LLM une requête de
recherche courte, en anglais technique, avec les termes exacts qu'on trouverait dans une datasheet. Le LLM sert ici
*aussi* à améliorer le retrieval. Comparez ensuite les chunks trouvés avec et sans réécriture.

In [13]:
def rewrite_query(question):
    messages = [
        {"role": "system", "content": "Transforme la question en une requête de recherche courte, en anglais "
         "technique, avec les termes exacts qu'on trouverait dans une datasheet de composant électronique. "
         "Réponds uniquement par la requête, sans guillemets ni commentaire."},
        {"role": "user", "content": question},
    ]
    return generate(messages)


question = "Quelle est la tension d'alimentation du HC-SR04 ?"
requete = rewrite_query(question)
print("question :", question, "\nrequête  :", requete)

print("\n--- recherche avec la question ---")
for c in search(question, chunks, embeddings, k=5):
    print(f"{c['score']:.3f}  {c['source']:<16} p.{c['page']}")
print("\n--- recherche avec la requête réécrite ---")
for c in search(requete, chunks, embeddings, k=5):
    print(f"{c['score']:.3f}  {c['source']:<16} p.{c['page']}")

question : Quelle est la tension d'alimentation du HC-SR04 ? 
requête  : Vcc supply voltage HC-SR04 datasheet

--- recherche avec la question ---
0.543  esp32.pdf        p.53
0.448  bme280.pdf       p.11
0.430  bme280.pdf       p.11
0.425  bme280.pdf       p.31
0.399  bme280.pdf       p.17

--- recherche avec la requête réécrite ---
0.624  esp32.pdf        p.51
0.612  arduino_uno.pdf  p.22
0.608  bme280.pdf       p.8
0.595  pico.pdf         p.21
0.588  pico.pdf         p.9


**Correction, exercice 3** — tout est dans le prompt (requête courte, vocabulaire de datasheet, *ne renvoyer que la
requête*). Résultat attendu : une requête du type `HC-SR04 supply voltage`, et le bon document qui remonte — la
traduction supprime l'écart de langue *et* rapproche le vocabulaire de celui du corpus. Coût : un appel LLM de plus
par question.

### Bonus — un RAG conversationnel

Pour que « et l'ESP32 ? » ait un sens après une question sur le Pico, il faut renvoyer au modèle **toute la discussion**
à chaque tour. Complétez `chat`. Question à se poser : que faire de l'historique quand il devient trop long pour la
fenêtre de contexte ?

In [14]:
historique = [{"role": "system", "content": SYSTEM_PROMPT}]  # le prompt système de l'exercice 1


def chat(message, k=settings.TOP_K):
    extraits = search(message, chunks, embeddings, k)
    extraits_texte = "\n\n".join(f"[{c['source']}, page {c['page']}]\n{c['text']}" for c in extraits)
    historique.append({"role": "user", "content": f"Extraits de documentation :\n\n{extraits_texte}\n\nQuestion : {message}"})
    reponse = generate(historique)
    historique.append({"role": "assistant", "content": reponse})
    print(reponse)


chat("Combien de broches GPIO possède le Raspberry Pi Pico ?")
print("\n" + "=" * 60 + "\n")
chat("et l'ESP32 ?")

26 broches GPIO.

Le Raspberry Pi Pico expose 26 des 30 GPIO possibles du RP2040 via ses broches. [pico.pdf, page 20]


Information absente des extraits fournis.


**Correction, bonus** — deux limites à discuter :

- « et l'ESP32 ? » est aussi la **requête de recherche** : seule, elle est pauvre. Amélioration classique : réécrire la
  question (exercice 3) *en tenant compte de l'historique* avant de chercher.
- L'historique enfle vite (les extraits de chaque tour y restent). Quand il approche la fenêtre de contexte : ne garder
  que les derniers tours, retirer les extraits des anciens messages, ou résumer le début de la discussion.

## Bilan

```
PDF ──► chunks ──► embeddings ──► index                     (indexation, TP1)
question ──► search ──► k chunks ──► prompt ──► LLM ──► réponse + sources   (TP1 + TP2)
```

Toutes les briques ont des réglages (découpage, modèle d'embeddings, k, hybride, prompt, modèle de génération…).
Pour choisir sans se fier à l'impression laissée par trois questions, il faut **mesurer** : c'est le TP3.